# Прогноз заездов в городском отеле
**Первая неделя · EDA, baseline и поиск аномалий**

Пока работаем с [H2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/) — данными городского отеля в Лиссабоне с плановыми заездами с июля 2015 по август 2017 года. Параллельно мы написали Нуно Антониу, автору и куратору набора [Hotel bookings data 2014–2023](https://zenodo.org/records/18119072), и запросили обезличенные записи по городским отелям C1 и C2. Для проекта подойдёт выгрузка хотя бы по одному отелю. Мы попросили включить плановую дату заезда, дату бронирования или LeadTime, итоговый статус и дату его присвоения, а также канал бронирования.

После ответа и получения доступа сверим определения полей и проверим, пересекаются ли записи с H2. Затем приведём данные к общей схеме и объединим их, сохранив признак отеля. Это позволит увеличить выборку и расширить временное окно анализа.

Для каждой даты заезда прогнозируем число заездов из броней, активных за 14, 7 или 3 дня до этой даты. Одна запись о заезде — одна бронь со статусом Check-Out. Считаем baseline по исторической доле заездов и сравниваем два способа поиска аномалий, IQR и Isolation Forest.


In [ ]:
import calendar
import hashlib
import io
import zipfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Image, Markdown, display
from sklearn.metrics import mean_absolute_error

from src.forecast import (
    detect_anomalies, forecast_baseline, make_snapshot, temporal_split,
)

SOURCE = 'https://ars.els-cdn.com/content/image/1-s2.0-S2352340918315191-mmc2.zip'
MIRROR = 'https://raw.githubusercontent.com/afaq-ahmad/Hotel-booking-demand-Case-Study/master/H2.csv'
H2_SHA256 = '5107090ffdc5585a9e18299801e9d85c3b459c235542882183a502a73ef63c4d'
HORIZONS = (14, 7, 3)
SEED = 42

def load_h2(path=Path('H2.csv')):
    """Проверяет, что загружен именно оригинальный H2."""
    if path.is_file():
        payload = path.read_bytes()
        source = path.name
    else:
        try:
            response = requests.get(SOURCE, timeout=30)
            response.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
                payload = archive.read('H2.csv')
            source = 'Приложение издателя mmc2.zip'
        except (requests.RequestException, zipfile.BadZipFile, KeyError):
            response = requests.get(MIRROR, timeout=45)
            response.raise_for_status()
            payload = response.content
            source = 'Зеркало H2.csv с проверкой SHA-256'
    payload = payload.replace(b'\r\n', b'\n')
    if hashlib.sha256(payload).hexdigest() != H2_SHA256:
        raise ValueError('Содержимое H2.csv отличается от версии издателя.')
    return pd.read_csv(
        io.BytesIO(payload), keep_default_na=False, na_values=['NA', ''],
        dtype={'Agent': 'string', 'Company': 'string', 'Country': 'string'}
    ), source

def show_figure(fig):
    output = io.BytesIO()
    fig.savefig(output, format='png', dpi=135, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    display(Image(data=output.getvalue()))

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'axes.axisbelow': True,
                     'grid.alpha': 0.22, 'figure.dpi': 110})
raw, source = load_h2()
assert raw.shape == (79_330, 31)
print(f'{source}. Размер H2: {len(raw):,} строк × {raw.shape[1]} столбец.')


## 1. Первичный обзор H2

Сначала выведем первые пять строк оригинального CSV, затем проверим типы полей, явные пропуски и значения NULL.


In [ ]:
with pd.option_context('display.max_columns', None):
    display(raw.head())

overview = pd.DataFrame({
    'Тип': raw.dtypes.astype(str),
    'Заполнено': raw.notna().sum(),
    'Явных пропусков': raw.isna().sum(),
    'Уникальных значений без пропусков': raw.nunique()
})
overview['Строковый NULL'] = 0
for column in ['Country', 'Agent', 'Company']:
    overview.loc[column, 'Строковый NULL'] = raw[column].str.strip().eq('NULL').sum()
display(overview)


## 2. Даты и статус бронирования

Дата создания = плановая дата заезда минус LeadTime. Используем срез **на конец календарного дня**. Бронь входит в когорту, если к этой дате она создана и её финальная отмена ещё не произошла. No-Show остаётся в когорте до заезда и даёт ноль в целевой переменной. Снимки приближённые: CSV содержит одну запись на бронь и дату последнего статуса, поэтому история переносов и промежуточных статусов неизвестна. Дни без активных броней включаем в оценку с фактом и прогнозом 0.

Финальные статусы и их даты нужны только для восстановления когорты и разметки. Изменяемые поля вроде длительности, числа гостей и ADR используем для диагностики. Канал DistributionChannel нужен для ретроспективного среза качества. Авторы использовали значения из журнала накануне заезда, когда они были доступны, поэтому для прогноза на ранних горизонтах берём календарь, LeadTime и признаки, которые вычисляются из восстановленной активной когорты.


In [ ]:
bookings = raw.copy()
months = {name: i for i, name in enumerate(calendar.month_name) if i}
bookings['arrival'] = pd.to_datetime({
    'year': bookings['ArrivalDateYear'],
    'month': bookings['ArrivalDateMonth'].map(months),
    'day': bookings['ArrivalDateDayOfMonth']
})
bookings['status_date'] = pd.to_datetime(bookings['ReservationStatusDate'], format='%Y-%m-%d')
bookings['created'] = bookings['arrival'] - pd.to_timedelta(bookings['LeadTime'], unit='D')
bookings['show'] = bookings['ReservationStatus'].eq('Check-Out').astype('int8')
bookings['canceled'] = bookings['ReservationStatus'].eq('Canceled')
bookings['nights'] = bookings['StaysInWeekendNights'] + bookings['StaysInWeekNights']
bookings['guests'] = bookings[['Adults', 'Children', 'Babies']].sum(axis=1, min_count=3)
bookings['ready'] = bookings[['arrival', 'status_date']].max(axis=1)
dates = pd.date_range(bookings['arrival'].min(), bookings['arrival'].max(), name='arrival')

assert bookings[['arrival', 'status_date', 'created']].notna().all().all()
assert bookings['ReservationStatus'].isin(['Check-Out', 'Canceled', 'No-Show']).all()
assert bookings['show'].eq(1 - bookings['IsCanceled']).all()
assert bookings['status_date'].ge(bookings['created']).all()
assert (bookings['status_date'].le(bookings['arrival']) | ~bookings['canceled']).all()
assert (bookings['status_date'].ge(bookings['arrival']) | bookings['canceled']).all()

status_counts = bookings['ReservationStatus'].value_counts()
audit = pd.Series({
    'Полные совпадения по 31 полю сверх первой строки': int(raw.duplicated().sum()),
    'Указано 0 гостей': int(bookings['guests'].eq(0).sum()),
    'Указано 0 ночей': int(bookings['nights'].eq(0).sum()),
    'Пропущено число детей': int(raw['Children'].isna().sum()),
    'Страна записана как NULL': int(raw['Country'].str.strip().eq('NULL').sum()),
    'Агент записан как NULL': int(raw['Agent'].str.strip().eq('NULL').sum()),
    'Компания записана как NULL': int(raw['Company'].str.strip().eq('NULL').sum())
}, name='Строк')
print(f'Заезды запланированы с {dates.min():%d.%m.%Y} по {dates.max():%d.%m.%Y} ({len(dates)} дат).')
display(status_counts.rename('Броней').to_frame())
display(audit.to_frame())
display(bookings.nlargest(5, 'ADR').round({'ADR': 2})[
    ['arrival', 'ADR', 'nights', 'guests', 'LeadTime', 'ReservationStatus']
])


Совпадающие строки сохраняем. В H2 нет ID брони, а совпадение полей совместимо с несколькими реальными бронями. По словарю статьи NULL в Agent и Company означает «не применимо». Нулевые ночи, нулевое число гостей и крайний ADR проверяем как кандидаты на ошибку или особый тип записи. Эти поля не влияют на вычисление когорты или прогноз. Для эксперимента с обработкой аномалий ниже используем признаки, которые доступны к дате среза.


## 3. Распределения и динамика бронирований

Посмотрим на числовые признаки и их хвосты, статус брони по месяцам, а также на объём и долю заездов по каналам. Здесь используем весь H2 для описания данных. Проверка гипотез о прогнозе ниже использует хронологическое разбиение.


In [ ]:
numeric = ['LeadTime', 'nights', 'guests', 'ADR', 'BookingChanges', 'DaysInWaitingList']
display(bookings[numeric].describe(percentiles=[.25, .5, .75, .95, .99]).round(2))

fig, axes = plt.subplots(2, 2, figsize=(11.6, 7.2), layout='constrained')
histograms = [
    ('LeadTime', np.arange(0, 651, 14), 'Срок бронирования', 'От бронирования до заезда, дни'),
    ('nights', np.arange(-.5, bookings['nights'].max() + 1.5), 'Длительность проживания', 'Ночей'),
    ('guests', np.arange(-.5, bookings['guests'].max() + 1.5), 'Размер брони', 'Гостей'),
]
for ax, (column, edges, title, label) in zip(axes.flat, histograms):
    ax.hist(bookings[column].dropna(), bins=edges, color='#277887', log=True)
    ax.set(title=title, xlabel=label, ylabel='Броней, лог. шкала')
adr_ax = axes[1, 1]
adr_cap = bookings['ADR'].quantile(.99)
adr_ax.hist(bookings.loc[bookings['ADR'].le(adr_cap), 'ADR'], bins=35, color='#c26d35')
adr_ax.set(title='ADR до 99-го процентиля', xlabel='Средняя стоимость номера за ночь',
           ylabel='Броней')
show_figure(fig)

monthly_input = bookings.assign(month=bookings['arrival'].dt.to_period('M'))
monthly = monthly_input.groupby(['month', 'ReservationStatus']).size().unstack(fill_value=0)
monthly.index = monthly.index.to_timestamp()
month_outcomes = [('Check-Out', 'Заезд', '#277887'), ('Canceled', 'Отмена', '#c26d35'),
                  ('No-Show', 'Неявка', '#76549b')]
raw_channels = bookings.groupby('DistributionChannel')['show'].agg(['size', 'mean'])
raw_channels = raw_channels.sort_values('size', ascending=False)
display(raw_channels.rename(columns={'size': 'Броней', 'mean': 'Доля заездов'}).round(3))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.1), layout='constrained',
                                  gridspec_kw={'width_ratios': [1.55, 1]})
for status, label, color in month_outcomes:
    left.plot(monthly.index, monthly[status], marker='o', markersize=2, label=label, color=color)
left.set(title='Итоги броней по месяцу планового заезда', xlabel='Месяц планового заезда',
         ylabel='Броней')
left.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
left.xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
left.legend(frameon=False)
channels = raw_channels.sort_values('size')
right.barh(channels.index, channels['size'], color='#277887')
right.set(title='Объём по каналам бронирования', xlabel='Броней, лог. шкала', xscale='log')
show_figure(fig)

print(f"Медиана LeadTime — {bookings['LeadTime'].median():.0f} дня, "
      f"99-й процентиль — {bookings['LeadTime'].quantile(.99):.0f}, "
      f"максимум — {bookings['LeadTime'].max()} дней.")
print(f"За пределами показанного диапазона ADR — "
      f"{bookings['ADR'].gt(adr_cap).sum()} броней, максимум — {bookings['ADR'].max():.0f}.")
print(f"На {raw_channels.index[0]} приходится {raw_channels['size'].iloc[0] / len(bookings):.1%} записей.")


## 4. Когорты на горизонтах 14, 7 и 3 дня

Сравним размеры когорт, долю заездов и чувствительность к выбору начала или конца календарного дня.


In [ ]:
snapshots = {h: make_snapshot(bookings, dates, h) for h in HORIZONS}
assert set(snapshots[14][0].index[snapshots[14][0]['show'].eq(1)]) <= set(
    snapshots[7][0].index[snapshots[7][0]['show'].eq(1)]
) <= set(snapshots[3][0].index[snapshots[3][0]['show'].eq(1)])

cohorts = pd.DataFrame([
    {'Горизонт, дни': h, 'Активные брони': len(active),
     'Заезды в когорте': int(active['show'].sum()),
     'Доля заездов, %': 100 * active['show'].mean(),
     'Даты без активных броней': daily['n'].eq(0).sum(),
     'Разница состава на начало и конец дня, броней': change}
    for h, (active, daily, change) in snapshots.items()
]).set_index('Горизонт, дни')
display(cohorts.round(2))


## 5. Временная валидация и EDA активных броней

Исторический период нужен для расчёта доли заездов, границ IQR и подгонки Isolation Forest. На следующем периоде смотрим устойчивость флагов. Финальный тест — с 1 мая по 31 августа 2017. На границах оставляем запас. Из исторического периода убираем даты, для которых итог хотя бы одной активной брони стал известен в день первого прогноза следующего периода или позже. Это учитывает и поздние выезды. Все варианты baseline оцениваем на одинаковых датах.


In [ ]:
WINDOWS = {
    'train': ('2015-07-01', '2016-12-15'),
    'validation': ('2017-01-01', '2017-02-14'),
    'test': ('2017-05-01', '2017-08-31'),
}

splits = {h: temporal_split(d, h, WINDOWS) for h, (_, d, _) in snapshots.items()}
assert all(len(split['test']) == 123 for split in splits.values())
split_rows = []
for h, (active, daily, _) in snapshots.items():
    for part, day_index in splits[h].items():
        split_rows.append({'Горизонт': h, 'Период': part, 'Дат': len(day_index),
                           'Броней': daily.loc[day_index, 'n'].sum(),
                           'Первая дата': day_index.min().date().isoformat(),
                           'Последняя дата': day_index.max().date().isoformat()})
display(pd.DataFrame(split_rows).set_index(['Горизонт', 'Период']))

# Разведочные срезы на обучении. Горизонт 7 дней показан подробно как средний из трёх.
active7, daily7, _ = snapshots[7]
train7 = active7.loc[active7['arrival'].isin(splits[7]['train'])]
eda7 = daily7.loc[splits[7]['train']]
lead_labels = ['7–13', '14–29', '30–59', '60–119', '120+']
lead_group = pd.cut(train7['LeadTime'], bins=[7, 14, 30, 60, 120, np.inf],
                    labels=lead_labels, right=False)
lead_summary = train7.groupby(lead_group, observed=True)['show'].agg(['size', 'mean'])
channel_eda = train7.groupby('DistributionChannel')['show'].agg(['size', 'mean']).sort_values('size', ascending=False)
display(channel_eda.rename(columns={'size': 'Активных броней', 'mean': 'Доля заездов'}).round(3))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.7), layout='constrained')
left.plot(eda7.index, eda7['n'].rolling(7, min_periods=1).mean(), label='Активные брони', color='#277887')
left.plot(eda7.index, eda7['actual'].rolling(7, min_periods=1).mean(), label='Заезды из них', color='#c26d35')
left.set(title='Когорта за 7 дней на обучении', ylabel='Число броней, среднее за 7 дат', xlabel='Дата планового заезда')
left.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
left.xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
left.legend(frameon=False)
bars = right.bar(lead_summary.index, lead_summary['mean'] * 100, color='#277887')
right.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
right.set(title='Доля заездов по сроку бронирования', xlabel='Дней от бронирования до заезда', ylabel='Заезды, %', ylim=(0, 100))
show_figure(fig)


**Гипотезы для дальнейших экспериментов.** Доля заездов может зависеть от срока бронирования, календаря и числа активных броней. В дни высокой концентрации канала исходы броней могут зависеть друг от друга. Выделенные детекторами записи могут быть настоящими редкими бронями. Проверим, что происходит с историческим baseline, если исключить их при расчёте доли заездов.


## 6. Статистический и ML-подходы к поиску аномалий

Сравниваем методы из лонгрида «Обнаружение аномалий» (разделы 2.1.1 и 2.2.4). **IQR** отмечает выброс за границами Q1 − 1,5×IQR и Q3 + 1,5×IQR хотя бы по одной координате. Он интерпретируем и устойчив к отдельным крайним значениям. **Isolation Forest** ищет необычные сочетания тех же координат. Его порог задаём по обучению с примерно такой же долей отмеченных броней, как у IQR. Сравниваем пересечение флагов, типичные примеры и результаты прогнозов. Разметки ошибок нет, поэтому согласие методов описывает их поведение, а точность прогнозирования проверяем экспериментом.

Оба метода видят срок бронирования, размер когорты и отклонение срока от дневной медианы. Границы и порог фиксируются по историческому периоду. Isolation Forest подгоняется без целевого признака, чтобы выделить необычные записи. Сравним количество флагов, их пересечение и конкретные примеры. Флаг означает необычное наблюдение и сам по себе не подтверждает ошибку данных.


In [ ]:
scored_snapshots = {}
anomaly_rows = []
for h, (active, _, _) in snapshots.items():
    scored, limits, _ = detect_anomalies(active, splits[h]['train'], seed=SEED)
    scored_snapshots[h] = scored
    if h == 7:
        limits7 = limits
    for part in ('train', 'validation', 'test'):
        cohort = scored.loc[scored['arrival'].isin(splits[h][part])]
        iqr, forest = cohort['iqr'], cohort['forest']
        anomaly_rows.append({
            'Горизонт': h, 'Период': part, 'Броней': len(cohort),
            'Только IQR': (iqr & ~forest).sum(), 'Только IF': (~iqr & forest).sum(),
            'Оба': (iqr & forest).sum(), 'Ни один': (~iqr & ~forest).sum(),
            'Отмечено IQR, %': 100 * iqr.mean(), 'Отмечено IF, %': 100 * forest.mean(),
            'Индекс Жаккара': (iqr & forest).sum() / max(1, (iqr | forest).sum())
        })
anomalies = pd.DataFrame(anomaly_rows).set_index(['Горизонт', 'Период'])
display(anomalies.round(3))
display(limits7.round(1))

inspect7 = scored_snapshots[7].loc[lambda d: d['arrival'].isin(splits[7]['train'])]
same_flags = inspect7['iqr'] & inspect7['forest']
example_cols = ['arrival', 'LeadTime', 'active_count', 'over_median_lead',
                'forest_score', 'nights', 'guests', 'ADR', 'ReservationStatus']
examples = pd.concat({
    'Только IQR': inspect7.loc[inspect7['iqr'] & ~inspect7['forest']].nlargest(4, 'LeadTime'),
    'Только IF': inspect7.loc[inspect7['forest'] & ~inspect7['iqr']].nlargest(4, 'forest_score'),
    'Оба': inspect7.loc[same_flags].nlargest(4, 'forest_score'),
})[example_cols]
display(examples.round({'forest_score': 3, 'ADR': 2}))

fig, ax = plt.subplots(figsize=(8.3, 4.1), layout='constrained')
for name, mask, color in [
    ('Без флага', ~inspect7['iqr'] & ~inspect7['forest'], '#b4bbc1'),
    ('Только IQR', inspect7['iqr'] & ~inspect7['forest'], '#277887'),
    ('Только IF', ~inspect7['iqr'] & inspect7['forest'], '#c26d35'),
    ('Оба метода', same_flags, '#76549b'),
]:
    data = inspect7.loc[mask]
    data = data.sample(n=min(700, len(data)), random_state=SEED)
    ax.scatter(data['LeadTime'], data['active_count'], s=12, alpha=.54, label=name, color=color)
ax.set(title='Какие брони отметили детекторы на горизонте 7 дней',
       xlabel='От бронирования до заезда, дни', ylabel='Активные брони на дату')
ax.legend(frameon=False)
show_figure(fig)


## 7. Исторический baseline и влияние флагов

Baseline умножает число активных броней на историческую долю заездов для этого горизонта. Проверим три способа расчёта доли: на всех исторических бронях, без флагов IQR и без флагов Isolation Forest. Сравним дневной MAE. Во всех трёх случаях прогнозируем одну и ту же полную когорту на каждой будущей дате и сравниваем с одним и тем же фактом. Так можно проверить последствия удаления из истории, сохраняя одинаковую задачу сравнения.


In [ ]:
rows = []
baseline_curves = {}
for h, (_, daily, _) in snapshots.items():
    indices = splits[h]
    variants = {'Все брони': None, 'Без флагов IQR': 'iqr', 'Без флагов IF': 'forest'}
    baseline_curves[h] = {}
    for name, flag in variants.items():
        rate, forecast, excluded = forecast_baseline(
            scored_snapshots[h], indices['train'], daily['n'], excluded_flag=flag
        )
        baseline_curves[h][name] = forecast
        rows.append({
            'Горизонт': h, 'История для расчёта доли': name,
            'Исключено из истории': excluded,
            'Историческая доля заездов, %': 100 * rate,
            'MAE на валидации': mean_absolute_error(
                daily.loc[indices['validation'], 'actual'], forecast.loc[indices['validation']]),
            'MAE на тесте': mean_absolute_error(
                daily.loc[indices['test'], 'actual'], forecast.loc[indices['test']])
        })
sensitivity = pd.DataFrame(rows).set_index(['Горизонт', 'История для расчёта доли'])
display(sensitivity.round(3))

fig, axes = plt.subplots(3, 1, figsize=(11.6, 7.4), sharex=True, layout='constrained')
for ax, h in zip(axes, HORIZONS):
    _, daily, _ = snapshots[h]
    index = splits[h]['test']
    ax.plot(index, daily.loc[index, 'actual'], color='#333b45', lw=1.25, label='Факт')
    ax.plot(index, baseline_curves[h]['Все брони'].loc[index], color='#277887',
            lw=1.25, label='Исторический baseline')
    ax.set(title=f'Горизонт {h} дн.', ylabel='Заезды')
axes[0].legend(frameon=False, ncol=2)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
axes[-1].set_xlabel('Дата планового заезда')
show_figure(fig)


In [ ]:
report = ['## Короткий отчёт для сдачи', '']
horizon_labels = {14: '14 дней', 7: '7 дней', 3: '3 дня'}

def ru(value, digits=2):
    return f'{value:,.{digits}f}'.replace(',', '\u202f').replace('.', ',')

report += [
    f"**Данные.** В H2 {ru(len(raw), 0)} броней и {raw.shape[1]} столбец. "
    f"Медиана срока бронирования — {ru(bookings['LeadTime'].median(), 0)} дня. "
    f"Доля заездов среди активных броней составляет "
    f"{ru(cohorts.loc[14, 'Доля заездов, %'], 1)}% за 14 дней и "
    f"{ru(cohorts.loc[3, 'Доля заездов, %'], 1)}% за 3 дня. "
    "Первичный обзор также выявил нулевое число гостей и ночей, а также редкие высокие значения ADR.",
    '',
    '**Сравнение детекторов.**'
]
ref = anomalies.loc[(7, 'train')]
report += [
    f"На исторических данных для горизонта 7 дней оба детектора отметили {ru(ref['Оба'], 0)} броней. "
    f"Только IQR отметил {ru(ref['Только IQR'], 0)}, только Isolation Forest — {ru(ref['Только IF'], 0)}. "
    f"Среди общих примеров есть реальные заезды с LeadTime {examples.loc['Оба', 'LeadTime'].iloc[0]} дня. "
    "IQR показывает, какое значение вышло за границы, а Isolation Forest выявляет необычные сочетания. "
    "Разметки ошибок нет, поэтому оцениваем совпадения и разбираем примеры.",
    '',
    "**Исторический baseline на 123 тестовых датах.** В каждом пункте указан MAE "
    "с полной историей, после исключения флагов IQR и после исключения флагов Isolation Forest.",
    ''
]
for h in HORIZONS:
    metrics = sensitivity.loc[h, 'MAE на тесте']
    report.append(f"- **{horizon_labels[h]}** — {ru(metrics.iloc[0])} / {ru(metrics.iloc[1])} / "
                  f"{ru(metrics.iloc[2])} заезда на дату.")
report += [
    '',
    '**Решение.** Сохраняем отмеченные брони в базовом наборе. Оба метода отмечают и фактические заезды, '
    'а исключение флагов из истории повысило тестовый MAE на всех трёх горизонтах. Используем флаги для разбора '
    'необычных случаев. Дальше проверим, насколько связь срока бронирования с долей заездов и '
    'влияние фильтрации устойчивы на других временных окнах.'
]
report_md = '\n'.join(report)
display(Markdown(report_md))
